In [177]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import plotly.offline as py
import plotly.graph_objs as go

In [178]:
data_dir = Path.cwd().parent / "data"

In [179]:
despesa_df = pd.read_parquet(data_dir / "despesa_ceaps.parquet")
despesa_df

,ANO,MES,SENADOR,TIPO_DESPESA,FORNECEDOR,DATA,DETALHAMENTO,VALOR_REEMBOLSADO
0,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",CENTRAIS ELÉTRICAS DE RONDÔNIA S.A. - CERON,2013-01-18,Nao preenchido,214.79
1,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",CENTRAIS ELÉTRICAS DE RONDÔNIA S.A. - CERON,2013-01-21,Nao preenchido,57.34
2,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",GILBERTO PISELO DO NASCIMENTO,2013-01-30,Nao preenchido,5000.00
3,2013,1,ACIR GURGACZ,"Aluguel de imóveis para escritório político, c...",OI S.A.,2013-01-14,Nao preenchido,398.10
4,2013,1,ACIR GURGACZ,"Locomoção, hospedagem, alimentação, combustíve...",YURI COMÉRCIO DE COMBUSTÍVEIS LTDA,2013-01-23,Nao preenchido,1128.00
...,...,...,...,...,...,...,...,...
225068,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-06,"Companhia Aérea: LATAM, Localizador: WIXHAI. P...",2893.04
225069,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-09,"Companhia Aérea: GOL, Localizador: WITOLM. Pas...",1180.19
225070,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-20,"Companhia Aérea: TAM, Localizador: THPKVQ. Pas...",2671.90
225071,2022,12,ZEQUINHA MARINHO,"Passagens aéreas, aquáticas e terrestres nacio...",Exceller Tour,2022-12-21,"Companhia Aérea: AZUL, Localizador: QNN9HX. Pa...",1334.31


In [180]:
despesa_ano = (despesa_df[['ANO', 'VALOR_REEMBOLSADO']]
               .groupby('ANO', as_index=False)
               .sum()
               .astype(int)
               .sort_values('ANO', ascending=True))
despesa_ano

,ANO,VALOR_REEMBOLSADO
0,2013,24824832
1,2014,22519394
2,2015,24859318
3,2016,25301959
4,2017,26672840
5,2018,25604701
6,2019,25248242
7,2020,20414923
8,2021,25020323
9,2022,27323316


# Despesas anuais

Analisando os gastos por ano, observamos que os valores mais baixos ocorreram em 2014 e 2020. Isso pode ser explicado pela recessão em 2014 e pela pandemia em 2020. Nos demais anos, os gastos se mantêm relativamente estáveis.

In [181]:
cores = ['#DD5E5E' if ano % 2 == 0 else '#1f77b4' for ano in despesa_ano['ANO']]

despesa = go.Bar(
    x=despesa_ano['ANO'], 
    y=despesa_ano['VALOR_REEMBOLSADO'],
    marker_color=cores,
    name='Despesas anuais', 
    hovertemplate='<b>Ano:</b> %{x}<br>' +
      '<b>Valor reembolsado:</b> R$ %{y:,.2f}<extra></extra>',
       showlegend=False)

layout = go.Layout(
    title='Despesas por ano', 
    xaxis=dict(title='Ano'), 
    yaxis=dict(title='Valor Reembolsado (R$)',
        showgrid=True,
        gridcolor='lightgray')
)

fig = go.Figure(data=[despesa], layout=layout)
fig.add_trace(go.Bar(
    x=[None], y=[None],
    marker_color='#DD5E5E', name='Ano eleitoral'
))
fig.add_trace(go.Bar(
    x=[None], y=[None],
    marker_color='#1f77b4', name='Ano não eleitoral'
))
fig.show()

In [182]:
ano_eleitoral = despesa_ano.loc[despesa_ano['ANO'] % 2 == 0, 'VALOR_REEMBOLSADO'].sum()
ano_nao_eleitoral = despesa_ano.loc[despesa_ano['ANO'] % 2 == 1, 'VALOR_REEMBOLSADO'].sum()

Ao separar anos eleitorais e não eleitorais, a diferença não é tão grande: os anos sem eleições registram gastos ligeiramente maiores. Isso sugere que crises específicas podem ter impacto maior do que o ano eleitoral no comportamento das despesas.

In [183]:
proportion = go.Pie(
    labels=['Ano eleitoral', 'Ano não eleitoral'],
    values=[ano_eleitoral, ano_nao_eleitoral],
    hovertemplate='<b>Ano:</b> %{label}<br>' +
                  '<b>Valor reembolsado:</b> R$ %{value:,.2f}<br>' +
                  '<b>Proporção:</b> %{percent}<extra></extra>')


fig = go.Figure(proportion)
fig.update_layout(title='Reembolsos por Tipo de Ano')
fig.show()

In [184]:
despesa_mes = (
    despesa_df[['MES', 'VALOR_REEMBOLSADO']]
    .groupby('MES', as_index=False)
    .sum()
    .astype(int)
    .sort_values('MES', ascending=True)
)
despesa_mes

,MES,VALOR_REEMBOLSADO
0,1,15478505
4,2,19390275
5,3,21624370
6,4,20621518
7,5,21119671
8,6,20246318
9,7,19574369
10,8,20360039
11,9,18969003
1,10,20950139


# Despesas por mês (agregado de todos os anos)

As despesas por mês mostram relativa estabilidade na maior parte do ano. Há uma tendência de aumento a partir de outubro, com pico em dezembro, possivelmente relacionado às festas de fim de ano. Janeiro registra valores mais baixos, refletindo férias e início/fim de mandatos.

In [185]:
despesa_mes = go.Bar(
    x=despesa_mes['MES'], 
    y=despesa_mes['VALOR_REEMBOLSADO'], 
    hovertemplate='<b>Mes:</b> %{x}<br>' +
      '<b>Valor reembolsado:</b> R$ %{y:,.2f}<extra></extra>')

layout = go.Layout(
    title='Despesas por mês', 
    xaxis=dict(title='Mês'), 
    yaxis=dict(title='Valor Reembolsado (R$)',
        showgrid=True,
        gridcolor='lightgray')
)

data = [despesa_mes]
fig = go.Figure(data=data, layout=layout)
fig.show()

In [186]:
despesa_mes_ano = (
    despesa_df[['MES', 'ANO', 'VALOR_REEMBOLSADO']]
    .groupby(['MES', 'ANO'], as_index=False)
    .sum()
    .astype(int)
    .sort_values('MES', ascending=True)
)

despesa_mes_ano

,MES,ANO,VALOR_REEMBOLSADO
0,1,2013,1480791
1,1,2014,1475224
2,1,2015,1445822
3,1,2016,1465053
4,1,2017,1560950
...,...,...,...
36,12,2019,2447587
35,12,2018,2869180
34,12,2017,2936752
33,12,2016,2652971


# Despesas mensais por ano

O gráfico de linhas por ano mostra que março apresenta maior gasto nos anos eleitorais, possivelmente ligado a campanhas e movimentações políticas.  
Já setembro tende a apresentar os menores valores, quando se aproximam as eleições.

In [187]:
fig7 = px.line(despesa_mes_ano, x='MES', y='VALOR_REEMBOLSADO', color='ANO', markers=True)
fig7.show()

In [188]:
senadores = (despesa_df[['SENADOR', 'VALOR_REEMBOLSADO']]
             .groupby('SENADOR', as_index=False)
             .sum()
             .sort_values('VALOR_REEMBOLSADO', ascending=False)
             )

senadores_maior= senadores.head(10)
senadores_menor = senadores.tail(10)

In [189]:
media_senadores = senadores_maior['VALOR_REEMBOLSADO'].mean()
media_senadores

np.float64(3765948.772)

# Senadores com maior gasto

Os 10 senadores com maiores despesas possuem média acima de R$ 3,7 milhões. Sérgio Petecão (PSD) lidera com R$ 4,2 milhões. Outros nomes de destaque incluem Fernando Collor, Ciro Nogueira e Davi Alcolumbre. 

Esses valores mostram que alguns senadores concentram parcela significativa dos gastos parlamentares.

In [190]:
senadores_maior = go.Bar(
    x = senadores_maior['SENADOR'],
    y = senadores_maior['VALOR_REEMBOLSADO'],
    hovertemplate='<b>Senador:</b> %{x}<br>' +
      '<b>Valor reembolsado:</b> R$ %{y:,.2f}<extra></extra>'
)

data = senadores_maior
fig = go.Figure(data=data)
fig.show()

# Senadores com menor gasto

Entre os 10 senadores com menor despesa, destacam-se José Sarney (MDB) com R$ 3,8 mil e Nailde Panta (Progressistas) com menos de R$ 1,7 mil.  

Essa diferença extrema evidencia grande disparidade no uso de recursos entre parlamentares.

In [191]:
senadores_menor = go.Bar(
    x = senadores_menor['SENADOR'],
    y = senadores_menor['VALOR_REEMBOLSADO'],
    hovertemplate='<b>Senador:</b> %{x}<br>' +
      '<b>Valor reembolsado:</b> R$ %{y:,.2f}<extra></extra>'
)

data = senadores_menor
fig = go.Figure(data=data)
fig.show()

In [192]:
mapeamento = {
    'Aluguel de imóveis para escritório político, compreendendo despesas concernentes a eles.': 'Aluguel de escritório',
    'Locomoção, hospedagem, alimentação, combustíveis e lubrificantes': 'Deslocamento e viagens',
    'Passagens aéreas, aquáticas e terrestres nacionais': 'Passagens nacionais',
    'Divulgação da atividade parlamentar': 'Divulgação parlamentar',
    'Aquisição de material de consumo para uso no escritório político, inclusive aquisição ou locação de software, despesas postais, aquisição de publicações, locação de móveis e de equipamentos. ': 'Material e escritório',
    'Contratação de consultorias, assessorias, pesquisas, trabalhos técnicos e outros serviços de apoio ao exercício do mandato parlamentar': 'Consultoria e serviços',
    'Serviços de Segurança Privada': 'Segurança privada'
}

despesa_df['TIPO_DESPESA'] = despesa_df['TIPO_DESPESA'].replace(mapeamento)

In [193]:
tipo_despesa = (despesa_df[['TIPO_DESPESA', 'VALOR_REEMBOLSADO']]
                .groupby('TIPO_DESPESA', as_index=False)
                .sum()
)
    
tipo_despesa['TIPO_DESPESA'] = tipo_despesa['TIPO_DESPESA'].apply(
    lambda x: x if len(x) <= 100 else x[:27] + '...'
)

# Reembolsos por tipo de despesa

As despesas com viagens representam 45,9% do total (somando passagens nacionais e custos de deslocamento).  
Os gastos relacionados à atividade parlamentar — incluindo aluguel de escritório, consultoria e material — correspondem a 39,87% do total.

Essa análise mostra que viagens e atividades parlamentares dominam a alocação de recursos.

In [194]:
tipo = go.Pie(
    labels=tipo_despesa['TIPO_DESPESA'],
    values=tipo_despesa['VALOR_REEMBOLSADO'],
    hovertemplate='<b>Ano:</b> %{label}<br>' +
                  '<b>Valor reembolsado:</b> R$ %{value:,.2f}<br>' +
                  '<b>Proporção:</b> %{percent}<extra></extra>')

fig = go.Figure(tipo)
fig.update_layout(title='Reembolsos por Tipo de Despesa')
fig.show()

# Conclusão

- Crises econômicas e sanitárias impactam mais os gastos do que o ano eleitoral.  
- Há padrões sazonais: picos em dezembro e quedas em janeiro.  
- A disparidade entre senadores é significativa, tanto nos maiores quanto nos menores gastos.  
- Viagens e despesas diretamente relacionadas à atividade parlamentar dominam os reembolsos.